# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 2 — Data Wrangling

**Estimated time:** 60 minutes

### Objective
Perform exploratory data analysis (EDA) on the Falcon 9 launch dataset and create a binary training label:

- `1` — the booster landed successfully.
- `0` — the booster did not land successfully.

Landing outcomes such as `True Ocean`, `True RTLS`, and `True ASDS` represent successful landings. Outcomes beginning with `False` represent unsuccessful landings. `None None` means no landing outcome was recorded.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 2. Load the SpaceX dataset

Load `dataset_part_1.csv` created in the previous lab. If the file is not in the notebook's current folder, the code downloads the official IBM course copy.

In [ ]:
from pathlib import Path

data_file = Path("dataset_part_1.csv")

if data_file.exists():
    df = pd.read_csv(data_file)
    print("Loaded local dataset_part_1.csv")
else:
    data_url = (
        "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
        "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
    )
    df = pd.read_csv(data_url)
    print("Loaded dataset from the IBM course data repository")

df.head(10)

## 3. Explore and understand the dataset

In [ ]:
# Display the number of rows and columns.
print("Dataset shape:", df.shape)

# Display basic information about the dataset.
df.info()

In [ ]:
# Summary statistics for numerical columns.
df.describe()

In [ ]:
# Calculate the percentage of missing values in each column.
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)

In [ ]:
# Identify numerical and categorical columns by data type.
print("Numerical columns:")
display(df.select_dtypes(include=['number']).columns.tolist())

print("Categorical / non-numerical columns:")
display(df.select_dtypes(exclude=['number']).columns.tolist())

# Display all column data types.
df.dtypes

## 4. Exploratory Data Analysis

### TASK 1: Calculate the number of launches on each launch site

Use `value_counts()` on the `LaunchSite` column.

In [ ]:
launches_per_site = df['LaunchSite'].value_counts()
display(launches_per_site)

In [ ]:
# Visualize the number of launches at each site.
launches_per_site.plot(kind='bar', figsize=(9, 5))
plt.xlabel('Launch Site')
plt.ylabel('Number of Launches')
plt.title('Number of Falcon 9 Launches by Launch Site')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### TASK 2: Calculate the number and frequency of launches for each orbit

Use `value_counts()` on the `Orbit` column.

In [ ]:
orbit_counts = df['Orbit'].value_counts()
display(orbit_counts)

In [ ]:
orbit_counts.plot(kind='bar', figsize=(9, 5))
plt.xlabel('Orbit')
plt.ylabel('Number of Launches')
plt.title('Number of Launches by Orbit')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### TASK 3: Calculate the number and frequency of mission outcomes

Inspect the `Outcome` column to understand the landing results.

In [ ]:
outcome_counts = df['Outcome'].value_counts(dropna=False)
display(outcome_counts)

### TASK 4: Explore payload mass by launch site

Compare payload mass for launches from different launch sites using a scatter plot.

In [ ]:
plt.figure(figsize=(10, 6))

for site in df['LaunchSite'].dropna().unique():
    site_data = df[df['LaunchSite'] == site]
    plt.scatter(
        site_data['FlightNumber'],
        site_data['PayloadMass'],
        label=site
    )

plt.xlabel('Flight Number')
plt.ylabel('Payload Mass (kg)')
plt.title('Payload Mass by Flight Number and Launch Site')
plt.legend()
plt.tight_layout()
plt.show()

### TASK 5: Calculate the success rate for each orbit

Create a temporary landing-success indicator from the `Outcome` field. A recorded outcome beginning with `True` is a successful landing; all other outcomes are treated as unsuccessful for this training label.

In [ ]:
df['LandingSuccess'] = df['Outcome'].apply(
    lambda outcome: 1 if str(outcome).strip().startswith('True') else 0
)

success_rate_by_orbit = df.groupby('Orbit')['LandingSuccess'].mean().sort_values(ascending=False)
display(success_rate_by_orbit)

In [ ]:
success_rate_by_orbit.plot(kind='bar', figsize=(9, 5))
plt.xlabel('Orbit')
plt.ylabel('Landing Success Rate')
plt.title('Landing Success Rate by Orbit')
plt.ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### TASK 6: Explore how landing success changes with flight number

Group flights into flight-number order and compare the landing success indicator.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['FlightNumber'], df['LandingSuccess'])
plt.xlabel('Flight Number')
plt.ylabel('Landing Success (0 = No, 1 = Yes)')
plt.title('Landing Success by Flight Number')
plt.yticks([0, 1])
plt.tight_layout()
plt.show()

## 5. Determine the training labels

The `Outcome` column contains values such as `True Ocean`, `False Ocean`, `True RTLS`, `False RTLS`, `True ASDS`, and `False ASDS`.

For supervised machine learning, convert these outcomes into a binary class:
- `1` when the outcome starts with `True`.
- `0` otherwise, including `False` and `None None`.

This follows the lab's definition of successful versus unsuccessful landing.

In [ ]:
# Create the binary Class label.
df['Class'] = df['Outcome'].apply(
    lambda outcome: 1 if str(outcome).strip().startswith('True') else 0
)

# Inspect the original outcomes and their corresponding labels.
display(df[['Outcome', 'Class']].head(15))

In [ ]:
# Count successful and unsuccessful landings.
class_counts = df['Class'].value_counts().sort_index()
class_counts.index = ['Unsuccessful (0)', 'Successful (1)']
display(class_counts)

In [ ]:
# Calculate the overall landing success rate.
landing_success_rate = df['Class'].mean() * 100
print(f"Overall landing success rate: {landing_success_rate:.2f}%")

In [ ]:
# Visualize the training labels.
class_counts.plot(kind='bar', figsize=(7, 5))
plt.xlabel('Landing Result')
plt.ylabel('Number of Launches')
plt.title('Falcon 9 First Stage Landing Results')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Prepare and save the final dataset

Save the dataset with the new `Class` training label. The resulting file can be used in the next machine-learning lab.

In [ ]:
# Save the wrangled dataset.
output_file = 'dataset_part_2.csv'
df.to_csv(output_file, index=False)

print(f"Saved {len(df)} records to {output_file}")
display(df.head())

## Lab 2 — Completed

In this lab, we:
1. Loaded the Falcon 9 launch dataset.
2. Checked dataset structure, data types, and missing values.
3. Counted launches by launch site and orbit.
4. Examined mission outcomes and payload mass.
5. Explored landing success patterns.
6. Created the binary training label `Class` (`1` = successful, `0` = unsuccessful).
7. Saved the final dataset as `dataset_part_2.csv`.

**Important:** Run all cells from top to bottom. Keep `dataset_part_1.csv` in the same folder if you want to use the file created in Lab 1; otherwise, the notebook will attempt to load the IBM-hosted dataset.